In [1]:
!pip install pyspark --quiet
print("PySpark installation complete!")

PySpark installation complete!


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import year, month, to_date, col, round as spark_round

import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

#Creating SparkSession
spark = SparkSession.builder \
        .appName('Day4_BigData_Sales')\
        .config('spark.sql.adaptive.enabled','true')\
        .getOrCreate()
print(f'Spark version : {spark.version}')
print(f'Spark App Name: {spark.sparkContext.appName}')
print(f'Spark Session : ACTIVE')

Spark version : 4.0.2
Spark App Name: Day4_BigData_Sales
Spark Session : ACTIVE


In [3]:
df_bronze = spark.read\
            .option('header','true')\
            .option('inferSchema','true')\
            .csv('/content/drive/MyDrive/Data_Engineering_Internship/large_sales_data.csv')
print("====BRONZE====")
print(f"Rows: {df_bronze.count()}")
print(f"Columns: {len(df_bronze.columns)}")
print()
df_bronze.printSchema()

====BRONZE====
Rows: 5000
Columns: 13

root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)



In [10]:
print("First 5 rows-")
df_bronze.show(5,truncate=False)

print("\nBasic statistics for numeric columns:")
df_bronze.select('quantity','unit_price','revenue').describe().show()

First 5 rows-
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|order_id|customer_name|product   |category   |quantity|unit_price|revenue|order_date|city     |region|sales_rep  |payment_method  |order_status|
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|1001    |Sneha Reddy  |Monitor   |Electronics|12      |22000     |264000 |2023-05-21|Mumbai   |West  |Meera Patel|UPI             |Delivered   |
|1002    |Ramesh Kumar |Printer   |Electronics|10      |12000     |120000 |2023-08-05|Delhi    |North |Anil Sharma|Credit Card     |Shipped     |
|1003    |Rahul Mishra |Mouse     |Accessories|10      |800       |8000   |2023-01-14|Ahmedabad|West  |Meera Patel|Cash on Delivery|Shipped     |
|1004    |Suresh Rao   |Tablet    |Electronics|5       |32000     |160000 |2023-01-04|Surat    |West  |Ravi Ku

In [14]:
df_bronze.write \
          .mode('overwrite') \
          .parquet('/content/drive/MyDrive/Data_Engineering_Internship/bronze_sales_data')

import os
def get_dir_size(path):
  if os.path.isfile(path):
    return os.path.getsize(path) /1024
  total = 0
  for dirpath, dirnames, filenames in os.walk(path):
    for f in filenames:
      total += os.path.getsize(os.path.join(dirpath, f)) /1024
  return total

csv_size = get_dir_size('/content/drive/MyDrive/Data_Engineering_Internship/large_sales_data.csv')
parquet_size = get_dir_size('/content/drive/MyDrive/Data_Engineering_Internship/bronze_sales_data')
reduction = (1 - parquet_size/csv_size) * 100

print(f"CSV file size: {csv_size:.2f} KB")
print(f"Parquet file size: {parquet_size:.2f} KB")
print(f"Reduction in size: {reduction:.2f}%")

CSV file size: 529.31 KB
Parquet file size: 55.10 KB
Reduction in size: 89.59%


In [25]:
#====SILVER====
df_silver = df_bronze \
            .dropDuplicates() \
            .dropna(subset=['order_id','product','revenue'])

df_silver = df_silver.withColumn(
    'order_date',
    to_date(col('order_date'),'yyyy-MM-dd')
)

df_silver = df_silver \
            .withColumn('order_year',year(col('order_date'))) \
            .withColumn('order_month',month(col('order_date')))

df_silver = df_silver.withColumn(
    'revenue_category',
    F.when(col('revenue')>40000,'High')
     .when(col('revenue')>10000,'Medium')
     .otherwise('Low')
)

print(f"Silver layer rows : {df_silver.count()}")
df_silver.select('product','revenue','order_month','order_year','revenue_category').show(5,truncate=False)

Silver layer rows : 5000
+--------+-------+-----------+----------+----------------+
|product |revenue|order_month|order_year|revenue_category|
+--------+-------+-----------+----------+----------------+
|Keyboard|13200  |2          |2023      |Medium          |
|Webcam  |17500  |1          |2023      |Medium          |
|Speaker |58500  |4          |2023      |High            |
|Keyboard|9600   |12         |2023      |Low             |
|Laptop  |180000 |8          |2023      |High            |
+--------+-------+-----------+----------+----------------+
only showing top 5 rows


In [26]:
df_silver.write \
      .mode('overwrite') \
      .parquet('/content/drive/MyDrive/Data_Engineering_Internship/silver_sales_data')

print(f'Silver Size : {get_dir_size("/content/drive/MyDrive/Data_Engineering_Internship/silver_sales_data"):.2f}KB')

df_verify = spark.read.parquet('/content/drive/MyDrive/Data_Engineering_Internship/silver_sales_data')
print(f"Read - back rows: {df_verify.count()}(Should match silver count)")
df_verify.printSchema()


Silver Size : 59.81KB
Read - back rows: 5000(Should match silver count)
root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_year: integer (nullable = true)
 |-- order_month: integer (nullable = true)
 |-- revenue_category: string (nullable = true)

